# Diagnóstico de Falhas em Rolamentos

**Disciplina:** Experimentação em Computação Aplicada

**Aluna:** Graziele Rodrigues

## Carregamento dataset em `.mat`

**Fonte oficial (usar esta referência na citação do TCC/relatório):**
Case Western Reserve University Bearing Data Center
https://engineering.case.edu/bearingdatacenter/download-data-file

O site separa os arquivos em 4 páginas. Para este projeto usamos apenas duas:
- **Normal Baseline Data** → arquivos da classe `Normal`
- **12k Drive End Bearing Fault Data** → arquivos das classes `Ball`, `InnerRace` e `OuterRace`, amostrados a 12 kHz no acelerômetro do lado de acionamento (Drive End)

Em cada página, os arquivos são organizados por **diâmetro do defeito** (0.007", 0.014", 0.021") e por **carga do motor** (0 a 3 HP). Os nomes no site são só números — não indicam a classe — então é aqui que mapeamos qual número corresponde a qual pasta, para a **condição de carga 1** (1 HP → 1772 RPM):

| Pasta de destino | Classe / diâmetro | Arquivo no site oficial |
|---|---|---|
| `Normal/` | Normal (baseline) | `98.mat` |
| `Ball/` | Falha de esfera — 0.007" | `119.mat` |
| `Ball/` | Falha de esfera — 0.014" | `186.mat` |
| `Ball/` | Falha de esfera — 0.021" | `223.mat` |
| `InnerRace/` | Falha na pista interna — 0.007" | `106.mat` |
| `InnerRace/` | Falha na pista interna — 0.014" | `170.mat` |
| `InnerRace/` | Falha na pista interna — 0.021" | `210.mat` |
| `OuterRace/` | Falha na pista externa — 0.007" (posição @6:00) | `131.mat` |
| `OuterRace/` | Falha na pista externa — 0.014" (posição @6:00) | `198.mat` |
| `OuterRace/` | Falha na pista externa — 0.021" (posição @6:00) | `235.mat` |

> **Sobre a posição da falha na pista externa:** o site disponibiliza 3 posições em relação à zona de carga — Centralizada (@6:00), Ortogonal (@3:00) e Opposta (@12:00) — mas só a Centralizada tem dado disponível nos três diâmetros. Por isso usamos apenas @6:00, que também é a posição mais adotada na literatura.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/CWRU_Data/Load_1'

Mounted at /content/drive


In [4]:
import os

mat_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith('.mat'):
            mat_files.append(os.path.join(root, f))

print(f"{len(mat_files)} arquivos .mat encontrados.")
for f in mat_files[:10]:
    print(" -", f)


10 arquivos .mat encontrados.
 - /content/drive/MyDrive/CWRU_Data/Load_1/Normal/normal_1772.mat
 - /content/drive/MyDrive/CWRU_Data/Load_1/Ball/ball_007_1772.mat
 - /content/drive/MyDrive/CWRU_Data/Load_1/Ball/ball_014_1772.mat
 - /content/drive/MyDrive/CWRU_Data/Load_1/Ball/ball_021_1772.mat
 - /content/drive/MyDrive/CWRU_Data/Load_1/InnerRace/ir_007_1772.mat
 - /content/drive/MyDrive/CWRU_Data/Load_1/InnerRace/ir_014_1772.mat
 - /content/drive/MyDrive/CWRU_Data/Load_1/InnerRace/ir_021_1772.mat
 - /content/drive/MyDrive/CWRU_Data/Load_1/OuterRace/or_007_1772.mat
 - /content/drive/MyDrive/CWRU_Data/Load_1/OuterRace/or_014_1772.mat
 - /content/drive/MyDrive/CWRU_Data/Load_1/OuterRace/or_021_1772.mat


## Pré-processamento para os modelos clássicos

Para cada arquivo `.mat`:
1. Carregamos o sinal do acelerômetro **Drive End (DE)** — a chave dentro do `.mat` termina em `DE_time`, mas o prefixo (`X097_...`, `X118_...`) varia por arquivo, então buscamos pela chave dinamicamente;
2. Fatiamos o sinal em janelas de tamanho fixo, com sobreposição (janelamento);
3. Para cada janela, extraímos características estatísticas no domínio do tempo;
4. O rótulo (classe) de cada janela é herdado da pasta em que o arquivo `.mat` está (`Normal`, `Ball`, `InnerRace`, `OuterRace`).
6. Normalizamos cada janela individualmente (z-score: subtrai a média e divide pelo desvio padrão da própria janela), já que a CNN é sensível à escala do sinal de entrada;
7. Damos ao array o formato exigido pela camada `Conv1D` do Keras: `(amostras, passos_de_tempo, canais)`;
8. Codificamos os rótulos como inteiros (a função de perda `sparse_categorical_crossentropy` trabalha diretamente com inteiros, sem precisar de one-hot encoding).

**Sobre o `form_factor`:** o cálculo correto é `RMS / média(|sinal|)` — usar a média do valor absoluto, não a média bruta do sinal (que pode ser próxima de zero ou negativa e distorcer a métrica, como identificado na análise exploratória anterior).

In [9]:
from scipy.io import loadmat
from scipy.stats import skew, kurtosis
import numpy as np

CLASSES = ["Normal", "Ball", "InnerRace", "OuterRace"]  # ajuste se suas pastas tiverem outros nomes

WINDOW_SIZE = 2048   # ~170 ms a 12 kHz — ajuste conforme necessário
OVERLAP = 0.5        # 50% de sobreposição entre janelas


def load_de_signal(filepath):
    """Carrega o vetor do acelerômetro Drive End de um arquivo .mat do CWRU."""
    mat = loadmat(filepath)
    de_keys = [k for k in mat.keys() if k.endswith('DE_time')]
    if not de_keys:
        return None
    return mat[de_keys[0]].squeeze().astype(float)


def infer_label(filepath, classes):
    """Extrai a classe a partir do nome da pasta que contém o arquivo."""
    parts = filepath.replace('\\', '/').split('/')
    for p in parts:
        if p in classes:
            return p
    return None


def extract_windows(signal, window_size=WINDOW_SIZE, overlap=OVERLAP):
    step = int(window_size * (1 - overlap))
    n = len(signal)
    return [signal[s:s + window_size] for s in range(0, n - window_size + 1, step)]


def compute_features(window):
    w = np.asarray(window, dtype=float)
    abs_w = np.abs(w)
    rms = np.sqrt(np.mean(w ** 2))
    mean_abs = np.mean(abs_w)
    return {
        'mean': np.mean(w),
        'std': np.std(w),
        'rms': rms,
        'max': np.max(w),
        'min': np.min(w),
        'skewness': skew(w),
        'kurtosis': kurtosis(w),
        'crest_factor': np.max(abs_w) / rms if rms != 0 else 0.0,
        'form_factor': rms / mean_abs if mean_abs != 0 else 0.0,
    }


In [10]:
import pandas as pd

records = []
raw_windows = []
skipped = []

for filepath in mat_files:
    label = infer_label(filepath, CLASSES)
    if label is None:
        skipped.append(filepath)
        continue

    signal = load_de_signal(filepath)
    if signal is None:
        skipped.append(filepath)
        continue

    for window in extract_windows(signal):
        feats = compute_features(window)
        feats['label'] = label
        feats['source_file'] = os.path.basename(filepath)
        records.append(feats)
        raw_windows.append(window)

df = pd.DataFrame(records)
raw_windows = np.array(raw_windows)  # shape: (n_janelas, WINDOW_SIZE)

print(f"Dataset final: {df.shape[0]} janelas, {df.shape[1] - 2} características.")
print("Distribuição de classes:")
print(df['label'].value_counts())

if skipped:
    print(f"\n{len(skipped)} arquivo(s) ignorado(s) (sem rótulo reconhecido ou sem canal DE):")
    for s in skipped:
        print(" -", s)

df.head()

Dataset final: 1529 janelas, 9 características.
Distribuição de classes:
label
Normal       471
OuterRace    354
Ball         352
InnerRace    352
Name: count, dtype: int64


,mean,std,rms,max,min,skewness,kurtosis,crest_factor,form_factor,label,source_file
0,0.008002,0.066445,0.066925,0.204234,-0.198810,-0.277955,-0.180225,3.051702,1.238296,Normal,normal_1772.mat
1,0.010276,0.065041,0.065848,0.204443,-0.197350,-0.225813,-0.086259,3.104787,1.247059,Normal,normal_1772.mat
2,0.009596,0.062987,0.063714,0.204443,-0.197350,-0.272366,-0.016090,3.208751,1.245645,Normal,normal_1772.mat
3,0.009945,0.063640,0.064412,0.204860,-0.182330,-0.200513,-0.191945,3.180459,1.232358,Normal,normal_1772.mat
4,0.010816,0.063052,0.063972,0.211327,-0.180244,-0.210519,-0.193607,3.303413,1.227519,Normal,normal_1772.mat


## Split treino / validação / teste

Cada arquivo `.mat` é uma gravação contínua e as janelas têm 50% de sobreposição. Um split aleatório por janela coloca no teste janelas que compartilham metade das amostras com janelas de treino (vazamento de dados), o que infla a métrica.

Por isso o split é feito em **blocos contíguos no tempo, dentro de cada arquivo** (70% treino, 15% validação, 15% teste), descartando as janelas de fronteira que se sobrepõem ao bloco vizinho. Todas as classes e todos os diâmetros de falha aparecem nos três conjuntos.

> **Limitação a declarar no relatório:** os três conjuntos vêm das mesmas gravações (mesma sessão e mesma carga). A métrica ainda é otimista quanto à generalização para falhas ou condições não vistas.


In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ============================================================
# Split em blocos temporais contíguos, por arquivo
# ============================================================

TRAIN_FRAC, VAL_FRAC = 0.70, 0.15
STEP = int(WINDOW_SIZE * (1 - OVERLAP))
GAP = int(np.ceil(1 / (1 - OVERLAP))) - 1   # janelas vizinhas que compartilham amostras (1 com 50% de overlap)

df["pos"] = df.groupby("source_file").cumcount()                 # posição temporal da janela no arquivo
n_win = df.groupby("source_file")["pos"].transform("size")
c1 = (TRAIN_FRAC * n_win).astype(int)
c2 = ((TRAIN_FRAC + VAL_FRAC) * n_win).astype(int)

idx_train = df.index[df["pos"] < c1 - GAP]                       # bloco inicial
idx_val   = df.index[(df["pos"] >= c1) & (df["pos"] < c2 - GAP)] # bloco do meio
idx_test  = df.index[df["pos"] >= c2]                            # bloco final

# Garantia: nenhum trecho do sinal é compartilhado entre os blocos
for f, g in df.groupby("source_file"):
    tr, va, te = (g.loc[g.index.intersection(ix), "pos"] for ix in (idx_train, idx_val, idx_test))
    assert tr.max() * STEP + WINDOW_SIZE <= va.min() * STEP, f
    assert va.max() * STEP + WINDOW_SIZE <= te.min() * STEP, f

y_train = df.loc[idx_train, "label"]
y_val   = df.loc[idx_val,   "label"]
y_test  = df.loc[idx_test,  "label"]

print(f"Treino: {len(idx_train)} | Validação: {len(idx_val)} | Teste: {len(idx_test)} janelas")
print(pd.DataFrame({"treino": y_train.value_counts(),
                    "validação": y_val.value_counts(),
                    "teste": y_test.value_counts()}))

# ============================================================
# Normalização individual das janelas para a CNN
# ============================================================

def normalize_window(window):
    """Z-score por janela: (x - média) / desvio padrão."""
    window = np.asarray(window, dtype=np.float32)
    mean = np.mean(window)
    std = np.std(window)

    if std > 1e-8:
        return (window - mean) / std

    # Janela constante
    return window - mean


raw_windows = np.asarray(raw_windows)

def to_cnn_input(idx):
    """Janelas normalizadas no formato (amostras, passos_de_tempo, canais) da Conv1D."""
    X = np.asarray([normalize_window(raw_windows[i]) for i in idx], dtype=np.float32)
    return X[..., np.newaxis]

X_train_cnn = to_cnn_input(idx_train)
X_val_cnn   = to_cnn_input(idx_val)
X_test_cnn  = to_cnn_input(idx_test)

# ============================================================
# Codificação dos rótulos
# ============================================================

label_encoder = LabelEncoder()
y_train_cnn = label_encoder.fit_transform(y_train)
y_val_cnn   = label_encoder.transform(y_val)
y_test_cnn  = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)

print("\nFormato para CNN 1D:")
print("X_train_cnn:", X_train_cnn.shape)
print("X_val_cnn  :", X_val_cnn.shape)
print("X_test_cnn :", X_test_cnn.shape)

print("\nClasses:")
for idx, class_name in enumerate(label_encoder.classes_):
    print(f"  {idx}: {class_name}")

# ============================================================
# Validações
# ============================================================

assert len(X_train_cnn) == len(y_train_cnn)
assert len(X_val_cnn) == len(y_val_cnn)
assert len(X_test_cnn) == len(y_test_cnn)
assert num_classes == len(CLASSES)


## Modelagem da CNN 1D

A arquitetura é intencionalmente compacta, já pensando na restrição de memória do Arduino Nano 33 BLE Sense, tendoo, então poucos filtros, `GlobalAveragePooling1D` no lugar de `Flatten` + `Dense` grande (reduz bastante o número de parâmetros) e `Dropout` para regularização, já que o dataset é relativamente pequeno.

In [14]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(42)

def build_cnn1d(input_shape, num_classes):
    return models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(16, kernel_size=7, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(32, kernel_size=5, activation='relu', padding='same'),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(32, kernel_size=3, activation='relu', padding='same'),
        layers.GlobalAveragePooling1D(),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])

cnn_model = build_cnn1d(input_shape=X_train_cnn.shape[1:], num_classes=num_classes)
cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
cnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 2048, 16)       │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 1024, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 1024, 32)       │         2,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 512, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 512, 32)        │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,012 (27.39 KB)

 Trainable params: 7,012 (27.39 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = cnn_model.fit(
    X_train_cnn, y_train_cnn,
    validation_data=(X_val_cnn, y_val_cnn),   # bloco de validação separado no tempo (sem sobreposição com o treino)
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1,
)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Treino')
axes[0].plot(history.history['val_loss'], label='Validação')
axes[0].set_title('Perda (loss)')
axes[0].set_xlabel('Época')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Treino')
axes[1].plot(history.history['val_accuracy'], label='Validação')
axes[1].set_title('Acurácia')
axes[1].set_xlabel('Época')
axes[1].legend()

plt.tight_layout()
plt.show()

## Avaliação preditiva da CNN 1D


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Define class_labels using the label_encoder from previous steps
class_labels = label_encoder.classes_

# Initialize summary_df if it doesn't exist (e.g., first run of this cell)
if 'summary_df' not in locals() or summary_df.empty:
    summary_df = pd.DataFrame(columns=['modelo', 'acuracia', 'f1_macro'])

y_pred_cnn_idx = cnn_model.predict(X_test_cnn).argmax(axis=1)
y_pred_cnn = label_encoder.inverse_transform(y_pred_cnn_idx)

acc_cnn = accuracy_score(y_test, y_pred_cnn)
f1_cnn = f1_score(y_test, y_pred_cnn, average='macro')

print("=== CNN 1D ===")
print(classification_report(y_test, y_pred_cnn, digits=3))

cm_cnn = confusion_matrix(y_test, y_pred_cnn, labels=class_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_cnn, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_labels, yticklabels=class_labels
)
plt.title('CNN 1D')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()


## Preparação para o Arduino Nano 33 BLE Sense

Comparação entre três versões do mesmo modelo, todas convertidas para TensorFlow Lite:

| Versão | Pesos | Ativações | Entrada/saída |
|---|---|---|---|
| FP32 | float32 | float32 | float32 |
| INT16 | int8 | int16 | int16 |
| INT8 | int8 | int8 | int8 |

**Pontos de atenção**

- O TFLite não tem pesos em 16 bits. A versão "INT16" é a **quantização 16x8** (ativações em 16 bits, pesos em 8 bits), e é assim que ela deve ser descrita no relatório.
- A calibração da quantização usa **apenas janelas de treino**. O teste nunca entra na calibração.
- Nas versões INT16 e INT8 a entrada e a saída são inteiras. No sketch, a janela normalizada precisa ser quantizada com o `scale` e o `zero_point` exportados em cada `.h`.
- A memória de ativações (arena do TFLite Micro) é uma **estimativa** por tempo de vida dos tensores. O valor real inclui alinhamento e buffers internos, então deve ser confirmado no dispositivo.
- Nas versões quantizadas a camada Dense usa pesos **por tensor** (não por canal). O kernel `FULLY_CONNECTED` do TFLite Micro da biblioteca Arduino não suporta pesos por canal, e a saída na placa divergiria do TFLite no PC.
- Confira se cada operador listado abaixo está disponível, no tipo de dado usado, na versão da biblioteca TFLite Micro que for usada no Arduino.


In [ ]:
import os
import numpy as np
import tensorflow as tf

EXPORT_DIR = os.path.join(os.path.dirname(DATA_DIR), 'arduino_export')
os.makedirs(EXPORT_DIR, exist_ok=True)

FLASH_KB, RAM_KB = 1024, 256   # Arduino Nano 33 BLE Sense (nRF52840)

# Keras 3 -> SavedModel (batch dinâmico vira batch = 1 no TFLite)
SAVED_MODEL_DIR = '/content/cnn_saved_model'
cnn_model.export(SAVED_MODEL_DIR)


def representative_dataset(n=300):
    """Calibração da quantização: somente dados de treino."""
    rng = np.random.default_rng(42)
    for i in rng.choice(len(X_train_cnn), size=min(n, len(X_train_cnn)), replace=False):
        yield [X_train_cnn[i:i + 1]]


def convert_tflite(variant):
    conv = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
    if variant == 'FP32':
        return conv.convert()

    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = representative_dataset
    # O kernel FULLY_CONNECTED do TFLite Micro (biblioteca Arduino) não trata pesos por canal:
    # sem isto, a saída na placa diverge do TFLite no PC.
    conv._experimental_disable_per_channel_quantization_for_dense_layers = True
    if variant == 'INT8':
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        io_type = tf.int8
    else:  # INT16 = quantização 16x8 (ativações int16, pesos int8)
        conv.target_spec.supported_ops = [
            tf.lite.OpsSet.EXPERIMENTAL_TFLITE_BUILTINS_ACTIVATIONS_INT16_WEIGHTS_INT8
        ]
        io_type = tf.int16
    conv.inference_input_type = io_type
    conv.inference_output_type = io_type
    return conv.convert()


VARIANTS = ['FP32', 'INT16', 'INT8']
tflite_models = {v: convert_tflite(v) for v in VARIANTS}

for v, b in tflite_models.items():
    path = os.path.join(EXPORT_DIR, f'modelo_{v.lower()}.tflite')
    with open(path, 'wb') as f:
        f.write(b)
    print(f"{v:>5}: {len(b) / 1024:6.1f} KB  ->  {path}")


In [ ]:
import warnings
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings('ignore', message='.*tf.lite.Interpreter is deprecated.*')

# Interpretador sem delegates (kernels de referência, mais próximo do que roda no microcontrolador)
def make_interpreter(model_bytes):
    it = tf.lite.Interpreter(
        model_content=model_bytes,
        experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES,
    )
    it.allocate_tensors()
    return it


def predict_tflite(model_bytes, X):
    """Inferência janela a janela, quantizando a entrada como o sketch fará."""
    it = make_interpreter(model_bytes)
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    scale, zero_point = inp['quantization']
    preds = []
    for x in X:
        xi = x[np.newaxis, ...]
        if inp['dtype'] != np.float32:
            info = np.iinfo(inp['dtype'])
            xi = np.clip(np.round(xi / scale) + zero_point, info.min, info.max)
        it.set_tensor(inp['index'], xi.astype(inp['dtype']))
        it.invoke()
        preds.append(int(it.get_tensor(out['index'])[0].argmax()))
    return np.array(preds)


def estimate_arena(interp, reshape_in_place=True):
    """Pico de memória (bytes) dos tensores de ativação, por tempo de vida.
    reshape_in_place=True : RESHAPE/EXPAND_DIMS reaproveitam o buffer da entrada (piso).
    reshape_in_place=False: cada um ganha um buffer próprio (teto).
    """
    ops = interp._get_ops_details()
    det = {d['index']: d for d in interp.get_tensor_details()}
    in_idx = {d['index'] for d in interp.get_input_details()}
    out_idx = {d['index'] for d in interp.get_output_details()}

    alias = {}
    if reshape_in_place:
        for op in ops:
            if op['op_name'] in ('RESHAPE', 'EXPAND_DIMS', 'SQUEEZE'):
                alias[op['outputs'][0]] = alias.get(op['inputs'][0], op['inputs'][0])
    root = lambda t: alias.get(t, t)

    acts = {root(t) for op in ops for t in op['outputs']} | in_idx   # pesos (constantes) ficam de fora
    first, last = {t: 0 for t in in_idx}, {}
    for k, op in enumerate(ops):
        for t in op['outputs']:
            first.setdefault(root(t), k)
            last[root(t)] = max(last.get(root(t), k), k)
        for t in op['inputs']:
            if root(t) in acts:
                last[root(t)] = max(last.get(root(t), k), k)
    for t in out_idx:
        last[root(t)] = len(ops)

    size = lambda t: int(np.prod(det[t]['shape'])) * np.dtype(det[t]['dtype']).itemsize
    return max(sum(size(t) for t in acts if first[t] <= k <= last[t]) for k in range(len(ops)))


# Referência: modelo Keras (float32) no mesmo bloco de teste
pred_keras = cnn_model.predict(X_test_cnn, verbose=0).argmax(axis=1)

rows = [{
    'modelo': 'Keras (referência)',
    'acuracia': accuracy_score(y_test_cnn, pred_keras),
    'f1_macro': f1_score(y_test_cnn, pred_keras, average='macro'),
}]
ops_por_modelo = {}

for v, b in tflite_models.items():
    it = make_interpreter(b)
    pred = predict_tflite(b, X_test_cnn)
    arena_min = estimate_arena(it, reshape_in_place=True)
    arena_max = estimate_arena(it, reshape_in_place=False)
    ops_por_modelo[v] = sorted({op['op_name'] for op in it._get_ops_details()})
    rows.append({
        'modelo': v,
        'acuracia': accuracy_score(y_test_cnn, pred),
        'f1_macro': f1_score(y_test_cnn, pred, average='macro'),
        'concordancia_keras': float(np.mean(pred == pred_keras)),
        'flash_KB': len(b) / 1024,
        'flash_%': 100 * len(b) / 1024 / FLASH_KB,
        'arena_min_KB': arena_min / 1024,
        'arena_max_KB': arena_max / 1024,
        'ram_%_max': 100 * arena_max / 1024 / RAM_KB,
    })

deploy_df = pd.DataFrame(rows)
pd.options.display.float_format = '{:.3f}'.format
display(deploy_df)

print("\nOperadores usados (para o resolver do TFLite Micro):")
for v, ops in ops_por_modelo.items():
    print(f"  {v:>5}: {', '.join(ops)}")

print("\nParâmetros de quantização de entrada/saída (usar no sketch):")
for v, b in tflite_models.items():
    it = make_interpreter(b)
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    print(f"  {v:>5}: entrada {np.dtype(inp['dtype']).name} scale={inp['quantization'][0]:.6g} zp={inp['quantization'][1]}"
          f" | saída {np.dtype(out['dtype']).name} scale={out['quantization'][0]:.6g} zp={out['quantization'][1]}")


In [ ]:
# ============================================================
# Exporta os modelos como arrays C (.h) e amostras de teste para o sketch
# ============================================================

def to_c_array(data, name, cols=12):
    lines = [', '.join(f'0x{b:02x}' for b in data[i:i + cols]) for i in range(0, len(data), cols)]
    body = ',\n  '.join(lines)
    return (f'alignas(16) const unsigned char {name}[] = {{\n  {body}\n}};\n'
            f'const unsigned int {name}_len = {len(data)};\n')


for v, b in tflite_models.items():
    it = make_interpreter(b)
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    name = f'modelo_{v.lower()}'
    header = (
        f'// {name}.h - gerado automaticamente ({v})\n'
        f'#pragma once\n\n'
        + to_c_array(b, name)
        + f'\nconst float {name}_in_scale  = {inp["quantization"][0]:.9e}f;\n'
        f'const int   {name}_in_zp     = {inp["quantization"][1]};\n'
        f'const float {name}_out_scale = {out["quantization"][0]:.9e}f;\n'
        f'const int   {name}_out_zp    = {out["quantization"][1]};\n'
    )
    with open(os.path.join(EXPORT_DIR, f'{name}.h'), 'w') as f:
        f.write(header)

# Uma janela de teste por classe (já normalizada), para validar a inferência no dispositivo
amostras = [int(np.where(y_test_cnn == c)[0][0]) for c in range(num_classes)]
X_amostras = X_test_cnn[amostras, :, 0]

rows_c = []
for w in X_amostras:
    vals = [f'{x:.7e}f' for x in w]
    rows_c.append('  {\n    ' + ',\n    '.join(', '.join(vals[i:i + 8]) for i in range(0, len(vals), 8)) + '\n  }')

classes_txt = ', '.join(f'{i}={c}' for i, c in enumerate(label_encoder.classes_))
with open(os.path.join(EXPORT_DIR, 'amostras_teste.h'), 'w') as f:
    f.write(f'// amostras_teste.h - uma janela de teste por classe ({classes_txt})\n#pragma once\n\n')
    f.write(f'#define N_AMOSTRAS {len(amostras)}\n#define TAM_JANELA {X_amostras.shape[1]}\n\n')
    f.write('const int rotulos_teste[N_AMOSTRAS] = {' + ', '.join(str(int(y_test_cnn[i])) for i in amostras) + '};\n\n')
    f.write('const float amostras_teste[N_AMOSTRAS][TAM_JANELA] = {\n' + ',\n'.join(rows_c) + '\n};\n')

print("Arquivos exportados em:", EXPORT_DIR)
for name in sorted(os.listdir(EXPORT_DIR)):
    print(f"  {name:<22} {os.path.getsize(os.path.join(EXPORT_DIR, name)) / 1024:8.1f} KB")


In [ ]:
# ============================================================
# Arquivos para o pipeline de medição na placa (run_benchmark.py)
# ============================================================
import json

np.save(os.path.join(EXPORT_DIR, 'X_test_cnn.npy'), X_test_cnn.astype(np.float32))
np.save(os.path.join(EXPORT_DIR, 'y_test_cnn.npy'), y_test_cnn.astype(np.int64))

meta = {'classes': [str(c) for c in label_encoder.classes_],
        'window_len': int(X_test_cnn.shape[1]),
        'variants': {}}

for v, b in tflite_models.items():
    # predições do TFLite no PC: referência para conferir a inferência feita na placa
    np.save(os.path.join(EXPORT_DIR, f'pred_tflite_{v.lower()}.npy'), predict_tflite(b, X_test_cnn))
    row = deploy_df[deploy_df['modelo'] == v].iloc[0]
    meta['variants'][v] = {
        'tflite_bytes': len(b),
        'arena_min_KB': float(row['arena_min_KB']),
        'arena_max_KB': float(row['arena_max_KB']),
        'host_accuracy': float(row['acuracia']),
        'host_f1_macro': float(row['f1_macro']),
    }

with open(os.path.join(EXPORT_DIR, 'meta.json'), 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("Pasta para levar ao PC do Arduino:", EXPORT_DIR)
for name in sorted(os.listdir(EXPORT_DIR)):
    print(f"  {name:<24} {os.path.getsize(os.path.join(EXPORT_DIR, name)) / 1024:8.1f} KB")
